In [1]:
# === Phase 2: Data Cleaning & Integration ===
# Loads the SQL extracts + both external CSVs as the starting point for cleaning

import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)  # make sure output folder exists

# --- Load from the database ---
con = sqlite3.connect(RAW / "ecommerce.db")
customers = pd.read_sql("SELECT * FROM customers", con)
products  = pd.read_sql("SELECT * FROM products", con)
orders    = pd.read_sql("SELECT * FROM orders", con)
order_items = pd.read_sql("SELECT * FROM order_items", con)
reviews   = pd.read_sql("SELECT * FROM reviews", con)
web_sessions = pd.read_sql("SELECT * FROM web_sessions", con)
con.close()

# --- Load external CSVs ---
legacy = pd.read_csv(RAW / "legacy_customers_export.csv")
catalog = pd.read_csv(RAW / "product_catalog_2024.csv")

print(customers.shape, products.shape, orders.shape, order_items.shape)
print(legacy.shape, catalog.shape)

(2500, 8) (300, 6) (9000, 5) (20362, 6)
(1427, 5) (267, 6)


In [2]:
# Legacy CSV has 4 date formats mixed in one column — normalize to one datetime dtype
# Strategy: try each known format against the whole column; combine_first fills in
# whichever rows the previous format failed to parse.

date_formats = [
    "%Y-%m-%d",       # 2024-03-10
    "%B %d, %Y",      # August 25, 2019
    "%m/%d/%Y",       # 01/25/2024
    "%d-%b-%Y",       # 23-Jul-2020
]

parsed = pd.Series(pd.NaT, index=legacy.index, dtype="datetime64[ns]")
for fmt in date_formats:
    # errors="coerce" turns anything that doesn't match this format into NaT
    attempt = pd.to_datetime(legacy["Signup_Dt"], format=fmt, errors="coerce")
    parsed = parsed.combine_first(attempt)  # keep first successful parse per row

legacy["signup_date"] = parsed

# Sanity check: how many rows failed to parse under ALL 4 formats?
print("Unparsed dates:", legacy["signup_date"].isna().sum())
print(legacy.loc[legacy["signup_date"].isna(), "Signup_Dt"].unique())  # inspect stragglers

Unparsed dates: 1
<StringArray>
[nan]
Length: 1, dtype: str


In [3]:
# Strip trailing whitespace from header names (the brief calls this out explicitly)
legacy.columns = legacy.columns.str.strip()

# Rename to match our internal schema naming so later merges are painless
legacy = legacy.rename(columns={
    "Customer Name": "name",
    "EMAIL_ADDR": "email",
    "Home City": "city",
    "Marketing Segment": "segment",
})

# --- Remove junk rows ---
# 1. Fully blank rows
before = len(legacy)
legacy = legacy.dropna(how="all")
# 2. Obvious test accounts (email or name containing 'test')
is_test = legacy["name"].str.contains("test", case=False, na=False) | \
          legacy["email"].str.contains("test", case=False, na=False)
legacy = legacy[~is_test]
print(f"Dropped {before - len(legacy)} junk/blank/test rows")

# --- Normalize casing so 'sarah robinson' and 'SARAH ROBINSON' match as duplicates ---
legacy["name_clean"] = legacy["name"].str.strip().str.title()
legacy["email_clean"] = legacy["email"].str.strip().str.lower()

# --- Detect exact duplicates (same email) ---
exact_dupes = legacy["email_clean"].dropna().duplicated().sum()
print(f"Exact duplicate emails: {exact_dupes}")

# Keep the most recently signed-up record when the same email appears more than once
# (arbitrary but documented decision — could also keep first; either is defensible
# as long as you state it in the report)
legacy = legacy.sort_values("signup_date").drop_duplicates(subset="email_clean", keep="last")

# --- Fuzzy near-duplicates: same name, no email to match on ---
# For rows missing email, flag same-name matches for manual review rather than
# auto-merging (safer since name collisions can be two different real people)
possible_fuzzy = legacy[legacy["email_clean"].isna()]
fuzzy_flagged = possible_fuzzy[possible_fuzzy.duplicated(subset="name_clean", keep=False)]
print(f"Flagged {len(fuzzy_flagged)} rows as possible fuzzy duplicates (no email to confirm)")

Dropped 2 junk/blank/test rows
Exact duplicate emails: 48
Flagged 0 rows as possible fuzzy duplicates (no email to confirm)


In [4]:
# === Missing value policy ===
# Each column gets a DIFFERENT rule based on what the missing value actually means
# and what it would cost you to get it wrong. No blanket fillna(0).

# --- age: ~6% missing. Impute with the MEDIAN, not mean (age is skewed by outliers,
# median is robust to that). Also add a flag column so downstream analysis can
# exclude imputed rows if needed (e.g. demographic breakdowns).
customers["age_was_missing"] = customers["age"].isna()
customers["age"] = customers["age"].fillna(customers["age"].median())

# --- city: ~3% missing. DROP is wrong here (loses otherwise-good order/revenue data);
# imputing a city would be fabricating a fact. Instead we FLAG it explicitly.
customers["city"] = customers["city"].fillna("Unknown")

# --- gender: missing + only 4 categories (F/M/Other/missing). Same logic as city —
# flag rather than guess, since gender can't be reasonably inferred.
customers["gender"] = customers["gender"].fillna("Unknown")

# --- review_text: ~20% missing. This is the one column we actually DROP the rows for
# — but only from a text-analysis subset, never from the main reviews table, because
# the rating itself is still valid and useful even without written text.
reviews_with_text = reviews.dropna(subset=["review_text"]).copy()
print(f"Reviews kept for text analysis: {len(reviews_with_text)} / {len(reviews)}")
# reviews (the full table, ratings intact) stays as-is — only used for rating-based analysis

Reviews kept for text analysis: 3162 / 4000


In [5]:
# IQR method: flag values outside 1.5x the interquartile range as outliers.
# Chose IQR over z-score because unit_price is heavily right-skewed (a few
# very expensive electronics) — z-score assumes roughly normal data, IQR doesn't.

Q1 = products["unit_price"].quantile(0.25)
Q3 = products["unit_price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

products["price_outlier"] = (products["unit_price"] < lower_bound) | (products["unit_price"] > upper_bound)
print(f"Flagged {products['price_outlier'].sum()} price outliers out of {len(products)} products")
print(products.loc[products["price_outlier"], ["product_id", "name", "unit_price"]])

# Decision: FLAG, don't drop or cap — these could be genuine premium products,
# not just data-entry errors, and you need them for revenue totals. The flag
# lets you optionally exclude them in specific analyses (e.g. "typical" pricing)
# while keeping them in revenue calculations.

Flagged 3 price outliers out of 300 products
     product_id                       name  unit_price
244         245   Throughout Personal Care     11506.5
249         250          Discover Haircare      8284.0
269         270  Environmental Non-Fiction     34872.0


In [7]:
# Rename catalog columns to match internal schema before merging
catalog_clean = catalog.rename(columns={
    "SKU": "product_id",
    "item_name": "catalog_name",
    "dept": "catalog_category",     # renamed to avoid colliding with products.category
    "list_price_usd": "catalog_price",
    "supplier_cost": "catalog_cost",
    "in_stock_units": "stock_units",
})

merged_products = products.merge(
    catalog_clean, on="product_id", how="outer", indicator=True
)
# no suffixes needed now — no more overlapping column names

db_only = merged_products[merged_products["_merge"] == "left_only"]
catalog_only = merged_products[merged_products["_merge"] == "right_only"]
print(f"Products only in DB: {len(db_only)}")
print(f"Products only in supplier catalog (not in DB at all): {len(catalog_only)}")
print(catalog_only[["product_id", "catalog_name", "catalog_category", "catalog_price"]])

Products only in DB: 45
Products only in supplier catalog (not in DB at all): 12
     product_id          catalog_name   catalog_category  catalog_price
300        9000       Small Prototype      Books & Media          38.48
301        9001    Standard Prototype    Beauty & Health          97.52
302        9002   Executive Prototype            Apparel          47.71
303        9003        Task Prototype  Sports & Outdoors         121.02
304        9004      Growth Prototype    Beauty & Health          75.01
305        9005    Suddenly Prototype     Home & Kitchen         174.90
306        9006  Individual Prototype            Apparel         288.34
307        9007      Leader Prototype  Sports & Outdoors         107.98
308        9008        Sign Prototype        Electronics         274.14
309        9009       Trial Prototype        Electronics         141.54
310        9010   Attention Prototype    Beauty & Health         157.72
311        9011         Boy Prototype  Sports & Outdoor

In [8]:
# Exact-duplicate detection: exclude the primary key (order_item_id is always unique
# even on duplicated rows), check duplication on the actual content columns instead
content_cols = ["order_id", "product_id", "quantity", "unit_price", "discount"]
dupe_mask = order_items.duplicated(subset=content_cols, keep="first")
print(f"Exact-duplicate order_items rows: {dupe_mask.sum()}")

order_items_clean = order_items[~dupe_mask].copy()

# Negative quantity = a return. Decision: KEEP them, and let them net against
# revenue naturally (a return should reduce recognized revenue) — this reflects
# real net revenue rather than overstating sales as if returns didn't happen.
# We add an explicit is_return flag so any analysis can isolate them if needed.
order_items_clean["is_return"] = order_items_clean["quantity"] < 0

order_items_clean["net_revenue"] = (
    order_items_clean["unit_price"]
    * order_items_clean["quantity"]
    * (1 - order_items_clean["discount"])
)

Exact-duplicate order_items rows: 186


In [9]:
# Join order_items -> orders (for date) -> products (for category)
oi_full = order_items_clean.merge(orders[["order_id", "order_date"]], on="order_id")
oi_full = oi_full.merge(products[["product_id", "category"]], on="product_id")

oi_full["order_month"] = pd.to_datetime(oi_full["order_date"]).dt.to_period("M").astype(str)

category_month_revenue = oi_full.pivot_table(
    values="net_revenue",
    index="category",
    columns="order_month",
    aggfunc="sum",
    fill_value=0,
)
category_month_revenue.head()

order_month,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,...,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
category,,,,,,,,,,,,,,,,,,,,,
Apparel,21239.8655,25533.6970,28150.8635,33055.0925,31137.7205,33209.5455,32224.3880,24778.2110,28286.5000,37086.6650,...,67746.3425,53113.8070,66473.6280,65667.8725,71784.9520,77192.5965,67390.5485,62906.6665,61550.3830,69400.1830
Beauty & Health,60052.7415,41534.9450,17577.9030,51036.4080,68534.6635,48307.8995,104647.7690,97695.6095,65168.2760,79944.8580,...,73444.4025,65487.6490,137785.5835,74673.4165,103049.2850,97775.5190,98815.7995,95329.4885,188833.1455,122731.3455
Books & Media,128047.8250,26458.4935,92026.2955,20088.4005,95527.8215,25385.1885,125698.5385,-10949.0165,57215.1645,88876.4215,...,152314.2495,159820.9260,129320.9505,193029.0195,137012.2540,205021.1805,150065.8715,96435.1925,161320.4735,98021.7455
Electronics,25582.7710,18903.1235,29059.6170,31505.1345,37199.9585,24216.8425,28442.8255,34811.9410,38387.8875,30239.7230,...,47790.5050,54830.4025,57836.6840,56161.3055,66704.2225,67158.3685,68340.3310,65971.6450,77072.5025,77909.7675
Home & Kitchen,26523.4810,20377.2125,25163.3990,14614.6935,16810.6400,26472.0930,27978.1685,28888.1615,28571.7130,31437.6050,...,42377.4225,41549.6695,65139.0805,48098.9375,64510.6775,69189.7875,42555.8425,61020.1155,53530.4700,54403.2910


In [10]:
# Weekly active customers using web_sessions, resampled on a datetime index
ws = web_sessions.copy()
ws["session_date"] = pd.to_datetime(ws["session_date"])
ws = ws.set_index("session_date")

weekly_active_customers = ws.resample("W")["customer_id"].nunique()
weekly_active_customers.name = "active_customers"
weekly_active_customers.head(10)

session_date
2023-01-01     20
2023-01-08    112
2023-01-15    106
2023-01-22    104
2023-01-29    108
2023-02-05     99
2023-02-12    109
2023-02-19    114
2023-02-26    121
2023-03-05    113
Freq: W-SUN, Name: active_customers, dtype: int64

In [11]:
# Save everything Phases 3 and 4 will read from — keeps the pipeline re-runnable
customers.to_csv(PROCESSED / "clean_customers.csv", index=False)
legacy.to_csv(PROCESSED / "clean_legacy_customers.csv", index=False)
merged_products.to_csv(PROCESSED / "clean_products.csv", index=False)
order_items_clean.to_csv(PROCESSED / "clean_order_items.csv", index=False)
reviews_with_text.to_csv(PROCESSED / "clean_reviews_with_text.csv", index=False)
reviews.to_csv(PROCESSED / "clean_reviews_full.csv", index=False)
category_month_revenue.to_csv(PROCESSED / "category_month_revenue.csv")
weekly_active_customers.to_csv(PROCESSED / "weekly_active_customers.csv")

print("Phase 2 outputs saved to data/processed/")

Phase 2 outputs saved to data/processed/
